# Naive RAG (HNSWLib + SQLite + Qwen3)

- hnswlib ANN index (persisted)
- SQLite metadata store (persisted)
- sentence-transformers embeddings
- Transformers generation on GPU



In [ ]:
!pip -q install -U \
  "torch" \
  "transformers>=4.44.0" \
  "accelerate>=0.33.0" \
  "sentence-transformers>=3.0.0" \
  "hnswlib>=0.8.0" \
  "numpy>=1.26.0" \
  "pypdf>=4.0.0" \
  "tqdm>=4.66.0"


In [1]:
import os, json, hashlib, sqlite3
from pathlib import Path
import numpy as np
from tqdm import tqdm

import hnswlib
import torch
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

from transformers import AutoTokenizer, AutoModelForCausalLM


## Paths


In [ ]:
# Project folders
BASE = Path(".")
DOCS_DIR = BASE / "docs"
DATA_DIR = BASE / "data"
DOCS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

INDEX_PATH = DATA_DIR / "index.bin"
DB_PATH    = DATA_DIR / "meta.sqlite"
CFG_PATH   = DATA_DIR / "index_config.json"

print("DOCS_DIR:", DOCS_DIR.resolve())
print("DATA_DIR:", DATA_DIR.resolve())


## Mounting Google Drive
Uncomment to store index/db in Drive.

In [3]:
# from google.colab import drive
# drive.mount("/content/drive")

# BASE = Path("/content/drive/MyDrive/rag_colab_hnsw")
# DOCS_DIR = BASE / "docs"
# DATA_DIR = BASE / "data"
# DOCS_DIR.mkdir(parents=True, exist_ok=True)
# DATA_DIR.mkdir(parents=True, exist_ok=True)

# INDEX_PATH = DATA_DIR / "index.bin"
# DB_PATH    = DATA_DIR / "meta.sqlite"
# CFG_PATH   = DATA_DIR / "index_config.json"

# print("DOCS_DIR:", DOCS_DIR.resolve())
# print("DATA_DIR:", DATA_DIR.resolve())


## Upload documents to `docs/`

In [ ]:
from google.colab import files

uploaded = files.upload()
for name, content in uploaded.items():
  (DOCS_DIR / name).write_bytes(content)

print("Docs in folder:")
for p in sorted(DOCS_DIR.iterdir()):
  print("-", p.name)


## Loaders + naive chunking

In [5]:
def read_pdf(path: Path) -> str:
    reader = PdfReader(str(path))
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)

def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def load_documents(docs_dir: Path):
    """Yield (source_path, full_text)."""
    for p in sorted(docs_dir.rglob("*")):
        if p.is_dir():
            continue
        suf = p.suffix.lower()
        if suf in {".txt", ".md"}:
            yield str(p), read_text(p)
        elif suf == ".pdf":
            yield str(p), read_pdf(p)

def naive_chunk(text: str, chunk_size: int = 900, overlap: int = 150):
    """Naive character chunking."""
    text = " ".join(text.split())
    out = []
    i = 0
    while i < len(text):
        out.append(text[i:i+chunk_size])
        i += max(1, chunk_size - overlap)
    return out

def stable_chunk_id(source: str, chunk_idx: int, text: str) -> str:
    s = f"{source}|{chunk_idx}|{text[:200]}"
    return hashlib.sha1(s.encode("utf-8")).hexdigest()


## SQLite helpers

In [6]:
def open_db(db_path: Path):
    conn = sqlite3.connect(str(db_path))
    conn.execute("""
      CREATE TABLE IF NOT EXISTS chunks (
        chunk_id TEXT PRIMARY KEY,
        label    INTEGER UNIQUE,
        source   TEXT,
        text     TEXT
      )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_chunks_label ON chunks(label)")
    conn.commit()
    return conn

def db_has_chunk(conn, chunk_id: str) -> bool:
    cur = conn.execute("SELECT 1 FROM chunks WHERE chunk_id=? LIMIT 1", (chunk_id,))
    return cur.fetchone() is not None

def db_get_by_label(conn, label: int):
    cur = conn.execute("SELECT chunk_id, source, text FROM chunks WHERE label=? LIMIT 1", (label,))
    return cur.fetchone()

def read_cfg(cfg_path: Path) -> dict:
    if cfg_path.exists():
        return json.loads(cfg_path.read_text())
    return {}

def write_cfg(cfg_path: Path, cfg: dict):
    cfg_path.write_text(json.dumps(cfg, indent=2))


## HNSW helpers

In [7]:
def load_or_create_hnsw(index_path: Path, cfg_path: Path, dim: int, space: str = "cosine"):
    """Returns (index, cfg)."""
    cfg = read_cfg(cfg_path)

    if index_path.exists() and cfg:
        index = hnswlib.Index(space=cfg.get("space", space), dim=int(cfg["dim"]))
        index.load_index(str(index_path))
        index.set_ef(int(cfg.get("ef", 64)))
        return index, cfg

    cfg = {
        "dim": dim,
        "space": space,
        "count": 0,
        "max_elements": 10000,
        "ef_construction": 200,
        "M": 16,
        "ef": 64,
    }
    index = hnswlib.Index(space=space, dim=dim)
    index.init_index(max_elements=cfg["max_elements"],
                     ef_construction=cfg["ef_construction"],
                     M=cfg["M"])
    index.set_ef(cfg["ef"])
    write_cfg(cfg_path, cfg)
    return index, cfg

def ensure_capacity(index, cfg: dict, cfg_path: Path, needed_total: int):
    max_el = int(cfg.get("max_elements", 10000))
    if needed_total <= max_el:
        return cfg
    new_max = max_el
    while new_max < needed_total:
        new_max *= 2
    index.resize_index(new_max)
    cfg["max_elements"] = new_max
    write_cfg(cfg_path, cfg)
    return cfg


## Embeddings model

In [ ]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)
DIM = embedder.get_sentence_embedding_dimension()
print("Embedding model:", EMBED_MODEL)
print("Embedding dim:", DIM)

def embed_texts(texts):
    # normalize for cosine space
    embs = embedder.encode(texts, normalize_embeddings=True)
    return np.asarray(embs, dtype=np.float32)


## Ingest (build/update index + SQLite)

In [ ]:
def ingest(docs_dir: Path):
    conn = open_db(DB_PATH)
    index, cfg = load_or_create_hnsw(INDEX_PATH, CFG_PATH, dim=DIM, space="cosine")

    all_chunks = []
    for source, text in load_documents(docs_dir):
        chunks = naive_chunk(text)
        for i, c in enumerate(chunks):
            cid = stable_chunk_id(source, i, c)
            all_chunks.append((cid, source, i, c))

    new_items = []
    for cid, source, idx, txt in all_chunks:
        if not db_has_chunk(conn, cid):
            new_items.append((cid, source, idx, txt))

    if not new_items:
        print("No new chunks to add.")
        conn.close()
        return

    start_label = int(cfg.get("count", 0))
    needed_total = start_label + len(new_items)
    cfg = ensure_capacity(index, cfg, CFG_PATH, needed_total)

    B = 256
    added = 0
    label_cursor = start_label

    for b0 in tqdm(range(0, len(new_items), B), desc="Ingest"):
        batch = new_items[b0:b0+B]
        texts = [x[3] for x in batch]
        vecs = embed_texts(texts)

        labels = np.arange(label_cursor, label_cursor + len(batch), dtype=np.int64)
        index.add_items(vecs, labels)

        conn.executemany(
            "INSERT INTO chunks(chunk_id, label, source, text) VALUES (?,?,?,?)",
            [(cid, int(lbl), src, txt) for (cid, src, _i, txt), lbl in zip(batch, labels)]
        )
        conn.commit()

        label_cursor += len(batch)
        added += len(batch)

    cfg["count"] = needed_total
    write_cfg(CFG_PATH, cfg)
    index.save_index(str(INDEX_PATH))
    conn.close()

    print(f"Done. Added {added} chunks.")
    print("Persisted:", INDEX_PATH, DB_PATH, CFG_PATH)

ingest(DOCS_DIR)


## Retriever (top-k)

In [10]:
def retrieve(query: str, k: int = 4):
    cfg = read_cfg(CFG_PATH)
    if not cfg or int(cfg.get("count", 0)) == 0:
        return []

    conn = open_db(DB_PATH)
    index = hnswlib.Index(space=cfg.get("space", "cosine"), dim=int(cfg["dim"]))
    index.load_index(str(INDEX_PATH))
    index.set_ef(int(cfg.get("ef", 64)))

    qv = embed_texts([query])
    k = min(k, int(cfg["count"]))
    labels, dists = index.knn_query(qv, k=k)
    labels = labels[0].tolist()
    dists = dists[0].tolist()

    hits = []
    for lbl, dist in zip(labels, dists):
        row = db_get_by_label(conn, int(lbl))
        if not row:
            continue
        chunk_id, source, text = row
        score = float(1.0 - dist)  # cosine distance -> rough similarity
        hits.append({"chunk_id": chunk_id, "source": source, "text": text, "score": score})

    conn.close()
    return hits


## Load Qwen3
Model: `Qwen/Qwen3-4B-Instruct-2507`

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print("Loaded:", MODEL_ID)


## Generate with stuffed context (naive RAG)

In [ ]:
@torch.inference_mode()
def generate_with_context(question: str, retrieved, max_new_tokens: int = 512, temperature: float = 0.2):
    if retrieved:
        context = "\n\n---\n\n".join(
            f"[source: {h['source']} | score: {h['score']:.3f}]\n{h['text']}"
            for h in retrieved
        )
    else:
        context = "(no context found)"

    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the provided CONTEXT to answer. If insufficient, say what is missing."},
        {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION:\n{question}"},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

def rag_answer(question: str, k: int = 4):
    hits = retrieve(question, k=k)
    answer = generate_with_context(question, hits)
    return hits, answer

# quick test
hits, ans = rag_answer("What are the key topics covered in my uploaded documents?")
print("Retrieved:")
for h in hits:
    print("-", h["source"], f"(score={h['score']:.3f})")
print("\nAnswer:\n", ans)


## Interactive chat loop

In [ ]:
while True:
    q = input("\n> ").strip()
    if q.lower() in {"exit", "quit"}:
        break
    if not q:
        continue
    hits, ans = rag_answer(q, k=4)
    print("\n--- retrieved ---")
    for h in hits:
        print(f"- {h['source']} (score={h['score']:.3f})")
    print("\n--- answer ---")
    print(ans)
